# Nudge Problem: Final evaluation

This notebook puts **every model side by side on the same test shoppers**, after the overnight tuning (`scripts/tune.py`). It follows **Week 3 s22** (keep validation and test decisions separate, report class-specific errors and costs) and **Week 8 s6** (is the complex model worth it?).

1. Read the tuning results: which settings were tried, and how much the result moves between seeds.
2. Pick **one** final setting per model, using **validation only**.
3. The comparison ladder on test: random, industry rule, logistic regression, RNN, Transformer.
4. Money on cart abandoners: discount € given to shoppers who came back anyway.
5. Robustness: train on October only, test on November.
6. The five failures, with real test visits.
7. What we can honestly say.

**Input:** `data/raw/visits_25pct.parquet`, `results/tuning/stage1.csv`, `results/tuning/stage2.csv`, the saved scores in `results/tuned/`.
**Output:** tables and charts below, saved as `results/evaluation_*.csv` and `results/evaluation_final.json`.

**Owner:** TBD (Comparison role)

## 0. Settings
Every path in one place. Section 5 **trains two models** (about 10–20 minutes on the Mac); set `RUN_OCT_NOV = False` to skip it.

In [ ]:
DATA = "../data/raw/visits_25pct.parquet"
TUNING = "../results/tuning"     # stage1.csv, stage2.csv (written by scripts/tune.py)
TUNED = "../results/tuned"       # <model>_<md5 of setting>_s<seed>_val.npy / _test.npy / .pt
OUT = "../results"
BOOT_N = 200                     # bootstrap draws (resampling shoppers)
RUN_OCT_NOV = True               # section 5 trains models

## 1. Import the packages
The same helpers as `02_baselines`, `03_rnn` and `04_transformer`. Two small new ones in `nudge/evaluate.py`: `industry_rule_score` (the rule as a score, with no random noise) and `top_share_rate` (come-back rate among the top 10%).

In [ ]:
import sys, pathlib, json, hashlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, so we can import nudge

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.metrics import ConfusionMatrixDisplay, average_precision_score
from IPython.display import display

from nudge.train import SEED, BLOCK_SIZE, VOCAB_SIZE, set_seed, visits_to_ids, word_counts, make_loader, train_model, predict_proba
from nudge.models import LogisticRegression
from nudge.transformer import VisitTransformer
from nudge.evaluate import pick_cutoff, quality, money, bootstrap, industry_rule_score, top_share_rate

set_seed(SEED)
torch.set_num_threads(4)   # keep the Mac responsive
TUNING, TUNED, OUT = pathlib.Path(TUNING), pathlib.Path(TUNED), pathlib.Path(OUT)

## 2. Load the visits
The same rules as every notebook: one row = one visit that ended without a purchase; **label 1** = the same shopper comes back and buys within 7 days, with no message. Split **by shopper** (80/10/10). The row order is the same as in `scripts/tune.py`, so the saved scores line up with these rows (checked below).

In [ ]:
visits = pd.read_parquet(DATA)
train_df = visits[visits["split"] == "train"].reset_index(drop=True)
val_df = visits[visits["split"] == "valid"].reset_index(drop=True)
test_df = visits[visits["split"] == "test"].reset_index(drop=True)
y = {"train": train_df["label"].to_numpy(), "val": val_df["label"].to_numpy(), "test": test_df["label"].to_numpy()}
base_rate = y["train"].mean()
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{name:10s} {len(df):9,} visits | {df['label'].mean():.1%} come back and buy")

## 3. What the tuning tried
**Stage 1:** every setting, trained on 25% of the training visits, one seed (42). **Stage 2:** the 2 best settings per model, retrained on **all** training visits with three seeds (42, 43, 44). Every score here is on **validation**; the test set was never used to choose anything (Week 3 s22).

The seed changes the starting weights and the order of the batches. If two settings differ by less than the spread between seeds, we cannot really tell them apart.

In [ ]:
stage1 = pd.read_csv(TUNING / "stage1.csv")
stage2 = pd.read_csv(TUNING / "stage2.csv")
NAMES = {"logistic_regression": "Logistic regression", "rnn": "RNN", "transformer": "Transformer"}
# The settings the notebooks used before tuning (02_baselines, 03_rnn, 04_transformer = workshop: d 32, 2 heads, 2 blocks, dropout 0.2)
WORKSHOP = {"logistic_regression": {"lr": 0.01, "weight_decay": 0.0},
            "rnn": {"d": 32, "hidden": 64, "dropout": 0.2, "lr": 0.001},
            "transformer": {"d": 32, "heads": 2, "blocks": 2, "dropout": 0.2, "lr": 0.001}}
WORKSHOP_KEY = {m: json.dumps(c, sort_keys=True) for m, c in WORKSHOP.items()}

for m, nice in NAMES.items():
    t = stage1[stage1["model"] == m].sort_values("val_pr_auc", ascending=False)
    t = t[["config", "val_pr_auc", "val_auc", "best_epoch", "seconds"]].reset_index(drop=True)
    t["note"] = np.where(t["config"] == WORKSHOP_KEY[m], "notebook setting", "")
    print(f"\nStage 1 · {nice} · {len(t)} settings (validation, 25% of training visits)")
    display(t.round(4))

In [ ]:
stage2_summary = (stage2.groupby(["model", "config"])
                  .agg(seeds=("seed", "count"), pr_auc_mean=("val_pr_auc", "mean"), pr_auc_std=("val_pr_auc", "std"),
                       pr_auc_min=("val_pr_auc", "min"), pr_auc_max=("val_pr_auc", "max"),
                       auc_mean=("val_auc", "mean"), auc_std=("val_auc", "std"), best_epoch_mean=("best_epoch", "mean"))
                  .reset_index().sort_values(["model", "pr_auc_mean"], ascending=[True, False]))
print("Stage 2 · all training visits · seeds 42, 43, 44 (validation)")
display(stage2_summary.round(4))

side = []
for m, nice in NAMES.items():
    s1 = stage1[stage1["model"] == m]
    best = s1.loc[s1["val_pr_auc"].idxmax()]
    ws = s1[s1["config"] == WORKSHOP_KEY[m]]
    side.append({"model": nice, "notebook setting": WORKSHOP_KEY[m],
                 "notebook PR-AUC (stage 1)": ws["val_pr_auc"].iloc[0] if len(ws) else np.nan,
                 "best setting": best["config"], "best PR-AUC (stage 1)": best["val_pr_auc"],
                 "gain": best["val_pr_auc"] - (ws["val_pr_auc"].iloc[0] if len(ws) else np.nan)})
print("Notebook setting vs tuned best (stage 1, same 25% of training visits, seed 42)")
pd.DataFrame(side).set_index("model").round(4)

## 4. One final setting per model
**Rule:** for each model, among the stage-2 settings that finished **all three seeds**, the one with the highest **mean validation PR-AUC**. Test is never looked at for this choice. The cell stops if a setting is incomplete, and it checks that the saved validation scores give back exactly the PR-AUC the tuning script wrote (so the rows are in the same order).

**Which scores we use on test: the average of the three seeds.** Each seed gives a slightly different model; averaging their probabilities is a small *ensemble*. It removes the luck of one seed, and it is the same thing the selection rule measured (the seed mean). The cut-off is also chosen on the averaged **validation** scores. The table after it shows each seed alone on test, so the spread is visible; the demo app uses one model (seed 42), whose row is in that table.

In [ ]:
complete = stage2_summary[stage2_summary["seeds"] == 3]          # only settings with all three seeds finished
assert set(complete["model"]) == set(NAMES), f"no complete setting yet for: {set(NAMES) - set(complete['model'])}"
final = complete.sort_values("pr_auc_mean", ascending=False).groupby("model").head(1).set_index("model")
final_runs, val_scores, test_scores = {}, {}, {}
for m in NAMES:
    key = final.loc[m, "config"]
    runs = stage2[(stage2["model"] == m) & (stage2["config"] == key)].sort_values("seed")
    assert sorted(runs["seed"]) == [42, 43, 44], f"{NAMES[m]}: seeds {list(runs['seed'])}, expected 42, 43, 44"
    assert all(r["tag"] == f"{m}_{hashlib.md5(key.encode()).hexdigest()[:8]}_s{r['seed']}" for _, r in runs.iterrows())
    final_runs[m] = runs
    val_scores[m] = {s: np.load(TUNED / f"{tag}_val.npy") for s, tag in zip(runs["seed"], runs["tag"])}
    test_scores[m] = {s: np.load(TUNED / f"{tag}_test.npy") for s, tag in zip(runs["seed"], runs["tag"])}
    assert all(len(v) == len(val_df) for v in val_scores[m].values()), "validation rows do not match tune.py"
    assert all(len(v) == len(test_df) for v in test_scores[m].values()), "test rows do not match tune.py"
    # Row alignment: the saved validation scores must give back the PR-AUC tune.py wrote for that run
    for _, r in runs.iterrows():
        recomputed = average_precision_score(y["val"], val_scores[m][r["seed"]])
        assert abs(recomputed - r["val_pr_auc"]) < 1e-4, f"{r['tag']}: PR-AUC {recomputed:.5f} vs stage2 {r['val_pr_auc']:.5f} (rows out of order?)"
    print(f"{NAMES[m]:20s} {key} | seeds {list(runs['seed'])} | validation PR-AUC {final.loc[m, 'pr_auc_mean']:.4f} ± {final.loc[m, 'pr_auc_std']:.4f}")

In [ ]:
per_seed = []
for m in NAMES:
    for s, sc in test_scores[m].items():
        q = quality(y["test"], sc, 0.5)
        per_seed.append({"model": NAMES[m], "seed": s, "test AUC": q["auc"], "test PR-AUC": q["pr_auc"]})
    avg = np.mean(list(test_scores[m].values()), axis=0)
    q = quality(y["test"], avg, 0.5)
    per_seed.append({"model": NAMES[m], "seed": "average", "test AUC": q["auc"], "test PR-AUC": q["pr_auc"]})
pd.DataFrame(per_seed).set_index(["model", "seed"]).round(4)

## 5. The comparison ladder on test
Five rungs, all on the **same** test visits:
- **Random:** a coin with the real buy rate: it says "comes back" for a share of visits equal to the training buy rate.
- **Industry rule:** left a cart → score 1, else 0 (Hostinger Reach, Shopify and Klaviyo treat every cart the same). In `02`/`04` tiny random noise broke the ties, so the rule's AUC moved between runs (0.556 vs 0.558). Now there is **no noise**: AUC counts a tie as half a win, so the number is the same on every run. Its cut-off is 0.5 (= "has a cart").
- **Logistic regression** on word counts (Week 3 s16–s18), **RNN** (Week 6 s28), **Transformer** (Week 8): the tuned final settings, seed-averaged. Each uses the cut-off with the best F1 on **validation** (`pick_cutoff`).

The 95% range comes from resampling test **shoppers** 200 times.

In [ ]:
rng = np.random.default_rng(SEED)
scores = {"Random": {"val": rng.uniform(size=len(val_df)), "test": rng.uniform(size=len(test_df))},
          "Industry rule": {"val": industry_rule_score(val_df), "test": industry_rule_score(test_df)}}
for m, nice in NAMES.items():
    scores[nice] = {"val": np.mean(list(val_scores[m].values()), axis=0), "test": np.mean(list(test_scores[m].values()), axis=0)}

cutoffs = {"Random": 1 - base_rate, "Industry rule": 0.5}
cutoffs.update({nice: pick_cutoff(y["val"], scores[nice]["val"]) for nice in NAMES.values()})

# A light copy of the test visits for the bootstrap (no word lists: copying them 200 times is slow)
frame = test_df[["user_id", "label", "decide_time", "cart_value", "next_order_value", "n_clicks"]].copy()
for name, s in scores.items():
    frame[name] = s["test"]

METRICS = ("auc", "pr_auc", "precision", "recall", "f1")
PAIRS = [("Transformer", "RNN"), ("Transformer", "Logistic regression"), ("RNN", "Logistic regression")]

def ladder(d):
    yy, out = d["label"].to_numpy(), {}
    for name in scores:
        q = quality(yy, d[name].to_numpy(), cutoffs[name])
        out.update({(name, k): q[k] for k in METRICS})
    for a, b in PAIRS:                       # paired: both models scored on the SAME resampled shoppers
        for k in ("auc", "pr_auc"):
            out[(f"{a} − {b}", k)] = out[(a, k)] - out[(b, k)]
    return out

point = ladder(frame)
ranges = bootstrap(frame, ladder, n=BOOT_N, seed=SEED)

quality_table = pd.DataFrame({k: {name: point[(name, k)] for name in scores} for k in METRICS})
quality_table.insert(0, "cut-off", pd.Series(cutoffs))
for k in ("auc", "pr_auc", "f1"):
    quality_table[f"{k} 95%"] = [f"{ranges[(n, k)][0]:.3f}–{ranges[(n, k)][1]:.3f}" for n in scores]
quality_table.round(3)

### Is the more complex model really better? Paired differences
Two overlapping ranges do not settle it. Each bootstrap draw scores **both** models on the **same** resampled shoppers and keeps the difference. If the 95% range of the difference stays above 0, the gain is not luck.

In [ ]:
diff_table = pd.DataFrame([{"comparison": f"{a} − {b}", "metric": k.upper().replace("_", "-"),
                            "difference": point[(f"{a} − {b}", k)],
                            "95% range": f"{ranges[(f'{a} − {b}', k)][0]:+.4f} to {ranges[(f'{a} − {b}', k)][1]:+.4f}",
                            "clearly better?": "yes" if ranges[(f"{a} − {b}", k)][0] > 0 else "no"}
                           for a, b in PAIRS for k in ("auc", "pr_auc")]).set_index(["comparison", "metric"])
diff_table.round(4)

### Confusion matrices
Rows = what really happened, columns = what the model said, at each model's validation cut-off (Week 3 s22: class-specific errors).

In [ ]:
fig, axes = plt.subplots(1, len(scores), figsize=(4 * len(scores), 3.8))
for ax, (name, s) in zip(axes, scores.items()):
    ConfusionMatrixDisplay.from_predictions(y["test"], s["test"] >= cutoffs[name], display_labels=["no", "comes back"],
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(name)
plt.tight_layout(); plt.show()

## 6. Money: who gets the 10% discount?
Only **cart abandoners**, the people today's flows discount. The industry rule discounts all of them. Each model ranks them and discounts the share it thinks is **least** likely to come back on its own. The score that marks "the 60% least likely" is taken from the **validation** cart abandoners, so test labels are never used. Random at the same share is the fair floor. Every policy: at most one discount per shopper per 7 days.

The share (80%, 60%, 40%) is set on validation **before** the one-per-week rule removes repeat discounts, so fewer discounts are actually sent than the share suggests; the "discounts" column shows the real count.
- **€ wasted** = 10% of what a shopper actually bought, for shoppers who got the discount and came back anyway.
- **Left without a nudge** = cart abandoners we skipped who did not come back. Whether a discount would have won them back is unknown: only an A/B test can say.

In [ ]:
SHARES = (0.8, 0.6, 0.4)
val_cart = val_df["cart_value"].to_numpy() > 0
policies = {"Industry rule (everyone)": np.ones(len(test_df), dtype=bool)}
for share in SHARES:
    for name in ("Random", *NAMES.values()):
        cut = np.quantile(scores[name]["val"][val_cart], share)
        policies[f"{name} ({share:.0%})"] = scores[name]["test"] < cut
for p, give in policies.items():
    frame[f"give: {p}"] = give

MONEY_KEYS = ("discounts_sent", "wasted_eur", "came_back_anyway_with_discount", "left_without_nudge")
money_numbers = lambda d: {(p, k): v for p in policies for k, v in money(d, d[f"give: {p}"].to_numpy()).items() if k in MONEY_KEYS}  # noqa: E731
money_point = money_numbers(frame)
money_ranges = bootstrap(frame, money_numbers, n=BOOT_N, seed=SEED)
money_table = pd.DataFrame([{"policy": p, "discounts": money_point[(p, "discounts_sent")],
                             "€ wasted": round(money_point[(p, "wasted_eur")]),
                             "€ wasted 95%": "{:,.0f}–{:,.0f}".format(*money_ranges[(p, "wasted_eur")]),
                             "came back anyway, with discount": money_point[(p, "came_back_anyway_with_discount")],
                             "left without a nudge": money_point[(p, "left_without_nudge")],
                             "left without 95%": "{:,.0f}–{:,.0f}".format(*money_ranges[(p, "left_without_nudge")])}
                            for p in policies]).set_index("policy")
money_table

In [ ]:
rule_waste = money_point[("Industry rule (everyone)", "wasted_eur")]
plot_rows = [(share, name, money_point[(f"{name} ({share:.0%})", "wasted_eur")], money_point[(f"{name} ({share:.0%})", "left_without_nudge")])
             for share in SHARES for name in ("Random", *NAMES.values())]
fig, ax = plt.subplots(figsize=(7, 4))
for name in ("Random", *NAMES.values()):
    pts = [(left, waste) for share, n, waste, left in plot_rows if n == name]
    ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", label=name)
ax.scatter([money_point[("Industry rule (everyone)", "left_without_nudge")]], [rule_waste], color="black", marker="s", zorder=3, label="Industry rule (everyone)")
ax.set(xlabel="cart abandoners left without a nudge (did not come back)", ylabel="€ discount given to shoppers who came back anyway",
       title="Less waste costs more people without a nudge (points: 80%, 60%, 40% discounted)")
ax.legend(); plt.tight_layout(); plt.show()

### Who is in the top 10%?
A simpler way to say "the model ranks well": among the 10% of test visits with the highest score, how many come back and buy, compared with the average? (When the 10% line falls inside a group of visits with the same score, that group counts in proportion, at its own come-back rate. This matters for the industry rule: every cart has score 1, and only about 8% of visits have a cart.)

In [ ]:
top_numbers = lambda d: {name: top_share_rate(d["label"].to_numpy(), d[name].to_numpy(), 0.10) for name in scores}  # noqa: E731
top_point = top_numbers(frame)
top_ranges = bootstrap(frame, top_numbers, n=BOOT_N, seed=SEED)
top_table = pd.DataFrame({"come-back rate in top 10%": top_point,
                          "95% range": {n: f"{lo:.1%}–{hi:.1%}" for n, (lo, hi) in top_ranges.items()},
                          "times the average": {n: v / y["test"].mean() for n, v in top_point.items()}})
print(f"Average come-back rate on test: {y['test'].mean():.1%}")
top_table.round(3)

## 7. Robustness: train on October, test on November
In the main split both months are mixed. A real shop trains on the past and uses the model on the future. November 2019 also had Black Friday (come-back rate 7.4% in October vs 9.3% in November). So: train on training shoppers' visits that **decide by 24 October**, stop early on validation visits from the same period, and score **test shoppers' November visits**.

"October" here means only **10 days of decisions, 15–24 October**: we score visits from 15 October on (1–14 October is history only), and a visit deciding after 24 October would have its 7-day label window reach into November.

Training on 10 days also means much **less data** than the main model, so a lower score could come from either cause. To separate them we train a **reference model**: the same setting, the same seed and the same number of epochs as the October model, on a **random sample of training visits from both months of the same size** (early stopping on a same-size random sample of validation visits).
- **Month-shift effect** = reference − October model: same amount of data, only the months differ.
- **Less-data effect** = main seed-42 model (all training visits, both months) − reference.

Only the final Transformer setting and logistic regression, **one seed (42)** each, to keep the Mac free. All three are scored on the same November test visits.

> **This cell trains four models (two per model type).** It only runs if `RUN_OCT_NOV = True` (section 0).

In [ ]:
if RUN_OCT_NOV:
    october = pd.Timestamp("2019-10-25")      # decide_time before this = decides 15–24 October
    tr = (train_df["decide_time"] < october).to_numpy()
    va = (val_df["decide_time"] < october).to_numpy()
    nov = (test_df["decide_time"] >= pd.Timestamp("2019-11-01")).to_numpy()
    sample_rng = np.random.default_rng(SEED)
    tr_ref = np.sort(sample_rng.choice(len(train_df), size=tr.sum(), replace=False))   # both months, same size
    va_ref = np.sort(sample_rng.choice(len(val_df), size=va.sum(), replace=False))
    print(f"October: train {tr.sum():,} | validation {va.sum():,}  ·  reference (both months): train {len(tr_ref):,} | validation {len(va_ref):,}")
    print(f"November test visits: {nov.sum():,} ({y['test'][nov].mean():.1%} come back)")

    train_sets = {"October only": (train_df["tokens"][tr], y["train"][tr], val_df["tokens"][va], y["val"][va]),
                  "both months, same size": (train_df["tokens"].iloc[tr_ref], y["train"][tr_ref], val_df["tokens"].iloc[va_ref], y["val"][va_ref])}
    test_ids = visits_to_ids(test_df["tokens"][nov])
    to_feats = lambda m, ids: word_counts(ids) if m == "logistic_regression" else ids   # noqa: E731
    oct_scores, epochs_used = {}, {}
    for m in ("logistic_regression", "transformer"):
        cfg = json.loads(final.loc[m, "config"])
        for when, (tr_tok, tr_y, va_tok, va_y) in train_sets.items():
            set_seed(SEED)
            model = (LogisticRegression(VOCAB_SIZE - 1) if m == "logistic_regression" else
                     VisitTransformer(VOCAB_SIZE, BLOCK_SIZE, d=cfg["d"], heads=cfg["heads"], blocks=cfg["blocks"], dropout=cfg["dropout"]))
            optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg.get("weight_decay", 0.0))
            # October model: the recipe of scripts/tune.py (up to 15 epochs, stop after 2 without improvement).
            # Reference: exactly as many epochs as the October model ran, so only the months differ.
            epochs, patience = (15, 2) if when == "October only" else (epochs_used[(m, "October only")], None)
            print(f"\n{NAMES[m]} · {when} · {cfg}")
            history = train_model(model, make_loader(to_feats(m, visits_to_ids(tr_tok)), tr_y, batch_size=512, shuffle=True),
                                  make_loader(to_feats(m, visits_to_ids(va_tok)), va_y, batch_size=4096), nn.BCEWithLogitsLoss(), optimizer,
                                  epochs=epochs, patience=patience)
            epochs_used[(m, when)] = len(history["train_loss"])
            oct_scores[(m, when)] = predict_proba(model, make_loader(to_feats(m, test_ids), y["test"][nov], batch_size=4096))

    WHEN = ("October only", "both months, same size", "both months, all visits")
    nov_frame = frame.loc[nov, ["user_id", "label"]].reset_index(drop=True)
    for m in ("logistic_regression", "transformer"):
        nov_frame[f"{m} · October only"] = oct_scores[(m, "October only")]
        nov_frame[f"{m} · both months, same size"] = oct_scores[(m, "both months, same size")]
        nov_frame[f"{m} · both months, all visits"] = test_scores[m][SEED][nov]    # the main seed-42 model
    def shift(d):
        yy, out = d["label"].to_numpy(), {}
        for m in ("logistic_regression", "transformer"):
            for when in WHEN:
                q = quality(yy, d[f"{m} · {when}"].to_numpy(), 0.5)
                out[(m, when, "auc")], out[(m, when, "pr_auc")] = q["auc"], q["pr_auc"]
            out[(m, "month shift", "auc")] = out[(m, "both months, same size", "auc")] - out[(m, "October only", "auc")]
            out[(m, "less data", "auc")] = out[(m, "both months, all visits", "auc")] - out[(m, "both months, same size", "auc")]
        return out
    shift_point, shift_ranges = shift(nov_frame), bootstrap(nov_frame, shift, n=BOOT_N, seed=SEED)
    oct_nov_table = pd.DataFrame([{"model": NAMES[m], "trained on": when, "epochs": epochs_used.get((m, when), "main run"),
                                   "November AUC": shift_point[(m, when, "auc")],
                                   "AUC 95%": "{:.3f}–{:.3f}".format(*shift_ranges[(m, when, "auc")]),
                                   "November PR-AUC": shift_point[(m, when, "pr_auc")]}
                                  for m in ("logistic_regression", "transformer") for when in WHEN]).set_index(["model", "trained on"])
    for m in ("logistic_regression", "transformer"):
        for effect in ("month shift", "less data"):
            lo, hi = shift_ranges[(m, effect, "auc")]
            print(f"{NAMES[m]:20s} AUC lost to {effect:11s} {shift_point[(m, effect, 'auc')]:+.4f} (95% range {lo:+.4f} to {hi:+.4f})")
    display(oct_nov_table.round(4))
else:
    oct_nov_table = None
    print("Skipped (RUN_OCT_NOV = False)")

## 8. Five failures, with real test visits
For each failure: the **Transformer's** AUC and PR-AUC on that slice next to all test visits, with 95% ranges by shopper. PR-AUC depends on how many positives a slice has (random PR-AUC = its come-back rate), so we also show **PR-AUC ÷ come-back rate** ("times better than random"). Then three real visits: the one the model was most wrongly sure **would** come back, the one it was most wrongly sure **would not**, and a typical one.

1. **One-click visits:** one click says almost nothing.
2. **Returning shoppers and history:** since round 2 the model gets history words (`bought_recently`, `visits_2_4`, `last_1_3d`, …). How much did they help?
3. **Not logged in = invisible:** shoppers with **only one scored visit in our data**. "Scored" means a visit that ended without a purchase between 15 October and 23 November, the visits we give a score to. These shoppers are mostly people we hardly know. Someone who browses on a phone and buys on a laptop without logging in looks like two different people to us, so their purchase is missing and they count as "did not come back".
4. **Very long, indecisive visits** (30 clicks or more).
5. **Expensive products** (any `_expensive` word): hesitation may look different when the price is high.

In [ ]:
MAIN = "Transformer"
tokens = test_df["tokens"]
visits_per_shopper = test_df.groupby("user_id")["visit"].transform("count")
SLICES = {"all test visits": np.ones(len(test_df), dtype=bool),
          "1. one click": (test_df["n_clicks"] == 1).to_numpy(),
          "2a. new shoppers": (tokens.str[0] == "new").to_numpy(),
          "2b. returning shoppers": (tokens.str[0] == "returning").to_numpy(),
          "3. only one scored visit in our data": (visits_per_shopper == 1).to_numpy(),
          "4. 30+ clicks": (test_df["n_clicks"] >= 30).to_numpy(),
          "5. any expensive product": tokens.map(lambda w: any("_expensive" in t for t in w)).to_numpy()}
for s, mask in SLICES.items():
    frame[f"slice: {s}"] = mask

def slice_numbers(d):
    out = {}
    for s in SLICES:
        part = d[d[f"slice: {s}"]]
        q = quality(part["label"].to_numpy(), part[MAIN].to_numpy(), cutoffs[MAIN])
        rate = part["label"].mean()
        out.update({(s, "come-back rate"): rate, (s, "auc"): q["auc"], (s, "pr_auc"): q["pr_auc"],
                    (s, "PR-AUC ÷ rate"): q["pr_auc"] / rate if rate else np.nan, (s, "recall"): q["recall"]})
    return out

slice_point = slice_numbers(frame)
slice_ranges = bootstrap(frame, slice_numbers, n=BOOT_N, seed=SEED)
failure_table = pd.DataFrame([{"slice": s, "visits": int(SLICES[s].sum()), "share": SLICES[s].mean(),
                               **{k: slice_point[(s, k)] for k in ("come-back rate", "auc", "pr_auc", "PR-AUC ÷ rate", "recall")},
                               "auc 95%": "{:.3f}–{:.3f}".format(*slice_ranges[(s, "auc")]),
                               "PR-AUC ÷ rate 95%": "{:.2f}–{:.2f}".format(*slice_ranges[(s, "PR-AUC ÷ rate")])}
                              for s in SLICES]).set_index("slice")
failure_table.round(3)

In [ ]:
def show_examples(slice_name, n_words=25):
    idx = np.where(SLICES[slice_name])[0]
    s, lab = scores[MAIN]["test"][idx], y["test"][idx]
    picks = []
    if (lab == 0).any(): picks.append(("most sure it WOULD come back, but did not", idx[lab == 0][np.argmax(s[lab == 0])]))
    if (lab == 1).any(): picks.append(("most sure it would NOT come back, but did", idx[lab == 1][np.argmin(s[lab == 1])]))
    picks.append(("a typical visit (median score)", idx[np.argsort(s)[len(s) // 2]]))
    print(f"=== {slice_name}")
    for why, i in picks:
        words = list(tokens[i])
        shown = " ".join(words[:n_words]) + (f" … ({len(words)} words)" if len(words) > n_words else "")
        print(f"  {why}: score {scores[MAIN]['test'][i]:.2f} | came back: {'yes' if y['test'][i] else 'no'}\n    {shown}")
    print()

for s in SLICES:
    if s != "all test visits":
        show_examples(s)

### Failure 2: did the history words help?
Round 1 (`docs/PROJECT_LOG.md` section 7) had no history words: only `new`/`returning` and the clicks. Its test numbers are below next to today's. **Careful:** three things changed at once (history words, 1 in 4 shoppers instead of 1 in 10, and tuned settings), and the test shoppers are not the same people, so this is a rough before/after, not a clean experiment. The fair part is the slice table above: the gap between new and returning shoppers.

In [ ]:
ROUND1 = {"Industry rule": (0.558, 0.124), "Logistic regression": (0.676, 0.157), "RNN": (0.679, 0.163), "Transformer": (0.682, 0.168)}
history_table = pd.DataFrame([{"model": n, "round 1 AUC": a, "now AUC": point[(n, "auc")], "AUC change": point[(n, "auc")] - a,
                               "round 1 PR-AUC": p, "now PR-AUC": point[(n, "pr_auc")]} for n, (a, p) in ROUND1.items()]).set_index("model")
history_table.round(3)

**One sentence per failure** (read the numbers above first):
1. **One click:** with one click there is almost no order to read, so every sequence model is reduced to the start word, the history words and one click; the slice AUC shows how close it gets to the overall number.
2. **History:** returning shoppers come back far more often; the history words let the model see *how* returning they are, but it still sees nothing outside the 14-day window.
3. **Invisible shoppers:** a shopper we see once has no history words, and if they bought on another device we labelled them "did not come back"; no model can fix a label we cannot see. In production, logged-in customers (as in our demo) remove most of this.
4. **Long visits:** long visits are rare and mixed (serious comparison or aimless scrolling); the model keeps only the last 64 words.
5. **Expensive products:** shoppers take longer over expensive items, so slow, repeated views mean something different there.

## 9. What we can honestly say
The sentences below are filled in by the next cell from the numbers computed above (nothing is typed by hand):

- On {n_test} test visits from shoppers never seen in training, the Transformer reaches AUC {tr_auc} ({tr_auc_lo}–{tr_auc_hi}); the RNN {rnn_auc}, logistic regression on word counts {lr_auc}, the industry rule {rule_auc}, random {rand_auc}.
- Transformer − RNN: {d_tr_rnn} AUC (95% {d_tr_rnn_lo} to {d_tr_rnn_hi}); Transformer − logistic regression: {d_tr_lr} ({d_tr_lr_lo} to {d_tr_lr_hi}). Order {order_verdict}.
- Discounting 60% of cart abandoners, the Transformer gives €{tr_waste} to shoppers who came back anyway, against €{rule_waste} under the industry rule; the cost is {tr_left} cart abandoners left without a nudge (rule: {rule_left}).
- Among the top 10% of scores, {tr_top} come back and buy, {tr_lift}× the average of {avg_rate}.
- On November visits, the Transformer loses {oct_shift} AUC to the month shift ({oct_shift_lo} to {oct_shift_hi}) and {oct_less} to having less data ({oct_less_lo} to {oct_less_hi}).
- Weakest slice: {weakest} (AUC {weakest_auc}).
- Nudge predicts who comes back without help. Whether the discount **causes** extra sales needs an A/B test, which we propose Hostinger runs next.

In [ ]:
r = lambda key, i: ranges[key][i]  # noqa: E731
f = {"n_test": f"{len(test_df):,}",
     "tr_auc": f"{point[('Transformer', 'auc')]:.3f}", "tr_auc_lo": f"{r(('Transformer', 'auc'), 0):.3f}", "tr_auc_hi": f"{r(('Transformer', 'auc'), 1):.3f}",
     "rnn_auc": f"{point[('RNN', 'auc')]:.3f}", "lr_auc": f"{point[('Logistic regression', 'auc')]:.3f}",
     "rule_auc": f"{point[('Industry rule', 'auc')]:.3f}", "rand_auc": f"{point[('Random', 'auc')]:.3f}"}
for short, (a, b) in {"d_tr_rnn": ("Transformer", "RNN"), "d_tr_lr": ("Transformer", "Logistic regression")}.items():
    k = (f"{a} − {b}", "auc")
    f[short], f[f"{short}_lo"], f[f"{short}_hi"] = f"{point[k]:+.4f}", f"{ranges[k][0]:+.4f}", f"{ranges[k][1]:+.4f}"
f["order_verdict"] = ("adds a small but real gain over counting words" if ranges[("Transformer − Logistic regression", "auc")][0] > 0
                      else "does not clearly beat counting words: an honest finding")
f.update({"tr_waste": f"{money_point[('Transformer (60%)', 'wasted_eur')]:,.0f}", "rule_waste": f"{rule_waste:,.0f}",
          "tr_left": f"{money_point[('Transformer (60%)', 'left_without_nudge')]:,}",
          "rule_left": f"{money_point[('Industry rule (everyone)', 'left_without_nudge')]:,}",
          "tr_top": f"{top_point['Transformer']:.1%}", "tr_lift": f"{top_point['Transformer'] / y['test'].mean():.1f}", "avg_rate": f"{y['test'].mean():.1%}"})
if oct_nov_table is not None:
    for short, effect in (("oct_shift", "month shift"), ("oct_less", "less data")):
        k = ("transformer", effect, "auc")
        f.update({short: f"{shift_point[k]:+.4f}", f"{short}_lo": f"{shift_ranges[k][0]:+.4f}", f"{short}_hi": f"{shift_ranges[k][1]:+.4f}"})
else:
    for short in ("oct_shift", "oct_less"):
        f.update({short: "(not run)", f"{short}_lo": "–", f"{short}_hi": "–"})
weakest = failure_table.drop("all test visits")["auc"].idxmin()
f.update({"weakest": weakest, "weakest_auc": f"{failure_table.loc[weakest, 'auc']:.3f}"})

SUMMARY = '''- On {n_test} test visits from shoppers never seen in training, the Transformer reaches AUC {tr_auc} ({tr_auc_lo}–{tr_auc_hi}); the RNN {rnn_auc}, logistic regression on word counts {lr_auc}, the industry rule {rule_auc}, random {rand_auc}.
- Transformer − RNN: {d_tr_rnn} AUC (95% {d_tr_rnn_lo} to {d_tr_rnn_hi}); Transformer − logistic regression: {d_tr_lr} ({d_tr_lr_lo} to {d_tr_lr_hi}). Order {order_verdict}.
- Discounting 60% of cart abandoners, the Transformer gives €{tr_waste} to shoppers who came back anyway, against €{rule_waste} under the industry rule; the cost is {tr_left} cart abandoners left without a nudge (rule: {rule_left}).
- Among the top 10% of scores, {tr_top} come back and buy, {tr_lift}× the average of {avg_rate}.
- On November visits, the Transformer loses {oct_shift} AUC to the month shift ({oct_shift_lo} to {oct_shift_hi}) and {oct_less} to having less data ({oct_less_lo} to {oct_less_hi}).
- Weakest slice: {weakest} (AUC {weakest_auc}).
- Nudge predicts who comes back without help. Whether the discount causes extra sales needs an A/B test, which we propose Hostinger runs next.'''
print(SUMMARY.format(**f))

## 10. Save the results for the report and the app

In [ ]:
quality_table.round(4).to_csv(OUT / "evaluation_quality.csv")
diff_table.round(4).to_csv(OUT / "evaluation_differences.csv")
money_table.to_csv(OUT / "evaluation_money.csv")
top_table.round(4).to_csv(OUT / "evaluation_top10.csv")
failure_table.round(4).to_csv(OUT / "evaluation_failures.csv")
if oct_nov_table is not None:
    oct_nov_table.round(4).to_csv(OUT / "evaluation_oct_nov.csv")
(OUT / "evaluation_final.json").write_text(json.dumps(
    {NAMES[m]: {"config": json.loads(final.loc[m, "config"]), "tags": list(final_runs[m]["tag"]),
                "val_pr_auc_mean": float(final.loc[m, "pr_auc_mean"]), "cutoff_seed_average": float(cutoffs[NAMES[m]])}
     for m in NAMES}, indent=2))
print("saved")

## 11. Connect the code to the course
| Idea | Where it appears here | Course |
|---|---|---|
| Choose on validation, test once | sections 3–5 | Week 3 s22, chapter 5 c10 |
| Class-specific errors, confusion matrix | section 5 | Week 3 s22 |
| Precision / recall trade-off as money | section 6 | Week 3 s22 (costs); `docs/course-map.md` section 4 |
| Logistic regression = perceptron + sigmoid | ladder rung 3 | Week 3 s16–s18 |
| RNN $H_t = \tanh(X_t W_{xh} + H_{t-1} W_{hh} + b_h)$ | ladder rung 4 | Week 6 s28 |
| Transformer encoder, attention $\text{softmax}(QK^\top/\sqrt{d})V$ | ladder rung 5 | Week 8 s17–s27 |
| Is the complex model worth it? | paired differences | Week 8 s6 |
| Training on the past, testing on the future | section 7 | not in the slides: `docs/course-map.md` gaps |

## Reflection questions
1. Why do we pick the final setting by validation PR-AUC and never by test, even though test is "the real number"?
2. Averaging three seeds is a small ensemble. What does it remove, and why does the demo app still use just one model?
3. The paired range for Transformer − RNN: does it include 0? What would you tell a Hostinger executive who asks "so is the Transformer better?"
4. Pick one of the five failures. Is it a problem of the model, of the data, or of the label? What data from Hostinger would fix it?